# **Demo: Role / Persona Priming & Context Grounding with Groq**

### **Steps to Follow:**
- **Step 1:** Set Up Groq Authentication and Select a Chat Model
- **Step 2:** Define a Parameterized Chat Completion Function
- **Step 3:** Baseline Evaluation (Unprimed Standard User Prompt)
- **Step 4:** Role Priming (Enforcing Senior SRE / Production-Ready Standards)
- **Step 5:** Context Grounding (Enforcing Architectural Constraints & Internal Tech Stacks)

### **Step 1: Set Up Groq Authentication and Select a Chat Model**
- Set `GROQ_API_KEY` in your environment or in Google Colab Secrets. If neither provides a key, enter it at the hidden prompt.
- Installs the OpenAI Python SDK and connects it to Groq. No OpenAI key or local GPU is required.
- Prefers `openai/gpt-oss-20b` when Groq lists it, then tries the other listed chat models in the preferred order. Set `GROQ_MODEL` to override the choice.

In [1]:
%pip install -q openai

import os
from getpass import getpass

from openai import OpenAI

groq_api_key = os.environ.get("GROQ_API_KEY")
if not groq_api_key:
    try:
        from google.colab import userdata
    except ImportError:
        pass
    else:
        try:
            groq_api_key = userdata.get("GROQ_API_KEY")
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass

if not groq_api_key:
    groq_api_key = getpass("Groq API key: ").strip()
if not groq_api_key:
    raise ValueError("A Groq API key is required.")

client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1",
)

available_models = {model.id for model in client.models.list().data}
preferred_models = (
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
)
groq_model = os.environ.get("GROQ_MODEL", "").strip()
if not groq_model:
    groq_model = next((name for name in preferred_models if name in available_models), "")
if groq_model not in available_models:
    listed_models = ", ".join(sorted(available_models)) or "none"
    raise ValueError(
        f"No usable Groq model selected ({groq_model or 'no preferred model listed'}). "
        f"Set GROQ_MODEL to a chat model from this list: {listed_models}"
    )

print(f"Groq client ready. Using model: {groq_model}")

Note: you may need to restart the kernel to use updated packages.
Groq client ready. Using model: openai/gpt-oss-20b


### **Step 2: Define a Parameterized Chat Completion Function**
- Accepts a `user_prompt` and an optional `system_prompt`.
- Sends the prompts to Groq as separate `user` and `system` messages.
- Uses `temperature=0` to reduce variability and returns the model's reply as text.

In [2]:
def get_chat_completion(user_prompt, system_prompt=None, max_tokens=220):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})

    options = {}
    if groq_model in ("openai/gpt-oss-20b", "openai/gpt-oss-120b"):
        options["reasoning_effort"] = "low"
    response = client.chat.completions.create(
        model=groq_model,
        messages=messages,
        max_completion_tokens=max_tokens,
        temperature=0,
        **options,
    )
    return (response.choices[0].message.content or "").strip()

print("Parameterized completion helper ready.")

Parameterized completion helper ready.


### **Step 3: Baseline Evaluation (Unprimed Standard User Prompt)**
- Queries the model with a common engineering request without any system role or technical grounding.
- **Observation:** Notice how models typically generate basic script-level code: missing typing, lacking retry backoffs, missing docstrings, and lacking enterprise resilience.

In [3]:
# Common developer prompt without system guidance
baseline_user_query = "Write a Python function to fetch data from an HTTP API endpoint with retry logic."

print("=== Baseline: Unprimed Model Output ===")
baseline_response = get_chat_completion(user_prompt=baseline_user_query, max_tokens=1800)
print(baseline_response)

=== Baseline: Unprimed Model Output ===
Below is a compact, production‑ready helper that

* uses the `requests` library (you can swap it for `httpx` or `urllib3` if you prefer),
* retries on network‑related errors and on HTTP status codes that are usually transient (5xx, 429),
* applies an exponential back‑off with jitter,
* allows you to customize the maximum number of attempts, the base delay, and the maximum delay,
* returns the parsed JSON (or raises a clear exception if the request ultimately fails).

```python
import time
import random
from typing import Any, Dict, Optional

import requests
from requests import Response, RequestException


class APIError(RuntimeError):
    """Raised when the API returns a non‑successful status code after all retries."""
    def __init__(self, status_code: int, url: str, body: Optional[str] = None):
        msg = f"API request to {url} failed with status {status_code}"
        if body:
            msg += f": {body}"
        super().__init__(msg)
 

### **Step 4: Role Priming (Enforcing Senior SRE / Production-Ready Standards)**
- **Role Priming** instructs the model to adopt a specific professional persona (e.g., *Staff Reliability & Security Engineer*).
- Forces the model to apply production standards: strict typing (`typing.Dict`), exponential backoff, explicit timeout definitions, and comprehensive error handling.

In [4]:
# Define a Staff SRE Persona
sre_system_prompt = """
You are a Staff Site Reliability and Python Backend Engineer.
Your objective is to write production-grade, hardened code.
Strict Requirements:
1. Always include Python type hints for all parameters and return types.
2. Implement exponential backoff for retries with an explicit timeout limit.
3. Use structured logging or error propagation, not silent passes or print statements.
4. Return only the production code and concise docstring, no conversational filler.
"""

print("=== Step 4: Role-Primed Output (Staff SRE Persona) ===")
primed_response = get_chat_completion(
    user_prompt=baseline_user_query,
    system_prompt=sre_system_prompt,
    max_tokens=2500
)
print(primed_response)

=== Step 4: Role-Primed Output (Staff SRE Persona) ===
```python
import logging
import time
from typing import Any, Dict, Optional

import requests
from requests.exceptions import RequestException

logger = logging.getLogger(__name__)


def fetch_with_retry(
    url: str,
    *,
    timeout: float = 10.0,
    max_retries: int = 5,
    backoff_factor: float = 0.5,
    headers: Optional[Dict[str, str]] = None,
) -> Dict[str, Any]:
    """
    Fetch JSON data from an HTTP API endpoint with exponential backoff retry logic.

    Parameters
    ----------
    url : str
        The API endpoint to request.
    timeout : float, optional
        Maximum time in seconds to wait for a single request.
    max_retries : int, optional
        Maximum number of retry attempts.
    backoff_factor : float, optional
        Multiplier for calculating the delay between retries.
    headers : dict, optional
        Optional HTTP headers to include in the request.

    Returns
    -------
    dict
        

### **Step 5: Context Grounding (Enforcing Architectural Constraints & Internal Tech Stacks)**
- **Context Grounding** binds the model to strict organizational constraints (e.g., prohibiting unauthorized external dependencies, restricting Python version, enforcing proprietary/standard libraries).
- In this scenario, we ground the model into an environment where external packages like `requests` or `urllib3` are **forbidden**, requiring the standard library's `urllib.request`.

In [5]:
# Context-Grounded System Prompt
grounded_system_prompt = """
You are a Senior Python Developer working in a zero-dependency, air-gapped environment.
Environment Constraints:
- Python 3.12 Standard Library ONLY.
- Third-party packages (like `requests`, `httpx`, or `urllib3`) are STRICTLY FORBIDDEN.
- You MUST use Python's built-in `urllib.request` and `json` modules.
- Wrap everything in a reusable function with PEP 8 compliance.
"""

grounded_user_query = "Write a helper function to send an HTTP GET request to an API and return the parsed JSON dictionary."

print("=== Step 5: Context-Grounded Output (Zero Third-Party Dependency Rule) ===")
grounded_response = get_chat_completion(
    user_prompt=grounded_user_query,
    system_prompt=grounded_system_prompt,
    max_tokens=2500
)
print(grounded_response)

=== Step 5: Context-Grounded Output (Zero Third-Party Dependency Rule) ===
```python
"""
Utility module for making HTTP GET requests and parsing JSON responses.

The function below is intentionally lightweight and uses only the
Python 3.12 standard library, making it suitable for a zero‑dependency,
air‑gapped environment.
"""

import json
import urllib.request
import urllib.error
from typing import Any, Dict, Optional


def get_json(url: str, *, timeout: Optional[float] = 10.0,
             headers: Optional[Dict[str, str]] = None) -> Dict[str, Any]:
    """
    Send an HTTP GET request to *url* and return the parsed JSON payload.

    Parameters
    ----------
    url : str
        The target URL to request.
    timeout : float, optional
        Timeout in seconds for the request. Defaults to 10.0 seconds.
    headers : dict[str, str], optional
        Optional HTTP headers to include in the request.

    Returns
    -------
    dict
        The JSON payload decoded into a Python dict